In [0]:
CATALOG = 'stock_research_capstone'
SCHEMA = 'main'

# Using Yahoo Finance (yfinance) - free, no API key needed
# Install if not already available
%pip install yfinance --quiet

TICKERS = ['AAPL', 'AMZN', 'NFLX', 'GOOGL', 'META', 'TSLA', 'NVDA', 'MSFT']

# Create catalog and schema
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
print(f'Target: {CATALOG}.{SCHEMA}')

In [0]:
import yfinance as yf
from datetime import date, timedelta
from pyspark.sql import functions as F

def fetch_price_history(ticker: str, days_back: int = 365) -> list:
    """Fetch daily OHLCV data for a ticker from Yahoo Finance."""
    end_date = date.today()
    start_date = end_date - timedelta(days=days_back)
    
    stock = yf.Ticker(ticker)
    hist = stock.history(start=start_date, end=end_date, interval="1d")
    
    # Convert DataFrame to list of dicts
    records = []
    for idx, row in hist.iterrows():
        records.append({
            "date": idx.strftime("%Y-%m-%d"),
            "open": float(row['Open']),
            "high": float(row['High']),
            "low": float(row['Low']),
            "close": float(row['Close']),
            "volume": int(row['Volume'])
        })
    return records


def fetch_company_profile(ticker: str) -> dict:
    """Fetch company fundamentals/profile from Yahoo Finance."""
    stock = yf.Ticker(ticker)
    info = stock.info
    
    return {
        "name": info.get('longName', ticker),
        "sector": info.get('sector', 'Unknown'),
        "industry": info.get('industry', 'Unknown'),
        "marketCap": info.get('marketCap', 0),
        "description": info.get('longBusinessSummary', ''),
        "website": info.get('website', ''),
        "employees": info.get('fullTimeEmployees', 0),
        "country": info.get('country', ''),
        "currency": info.get('currency', 'USD')
    }


def fetch_news(ticker: str, limit: int = 20) -> list:
    """Fetch recent news articles for a ticker from Yahoo Finance."""
    stock = yf.Ticker(ticker)
    news = stock.news[:limit] if hasattr(stock, 'news') else []
    
    articles = []
    for item in news:
        articles.append({
            "title": item.get('title', ''),
            "source": item.get('publisher', 'Yahoo Finance'),
            "publishedAt": item.get('providerPublishTime', 0),
            "url": item.get('link', ''),
            "summary": item.get('title', ''),  # Yahoo doesn't provide summary
            "content": item.get('title', '')
        })
    return articles


print("Yahoo Finance helper functions defined.")



In [0]:

# Collect raw price data for all tickers
all_prices = []
for ticker in TICKERS:
    try:
        prices = fetch_price_history(ticker, days_back=365)
        for p in prices:
            p["ticker"] = ticker
        all_prices.extend(prices)
        print(f" ✓ {ticker}: {len(prices)} records")
    except Exception as e:
        print(f" X {ticker}: {e}")

# Write raw JSON as Bronze table
if all_prices:
    df_bronze_prices = spark.createDataFrame(all_prices)
    df_bronze_prices.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.bronze_prices")
    print(f"\nBronze prices: {len(all_prices)} rows written to {CATALOG}.{SCHEMA}.bronze_prices")
else:
    print("No price data fetched.")



In [0]:

# Collect company profiles
all_companies = []
for ticker in TICKERS:
    try:
        profile = fetch_company_profile(ticker)
        profile["ticker"] = ticker
        all_companies.append(profile)
        print(f" ✓ {ticker}")
    except Exception as e:
        print(f" X {ticker}: {e}")

if all_companies:
    df_bronze_companies = spark.createDataFrame(all_companies)
    df_bronze_companies.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.bronze_companies")
    print(f"\nBronze companies: {len(all_companies)} rows written")



In [0]:
# Collect news articles
all_news = []
for ticker in TICKERS:
    try:
        articles = fetch_news(ticker, limit=50)
        for a in articles:
            a["ticker"] = ticker
        all_news.extend(articles)
        print(f" ✓ {ticker}: {len(articles)} articles")
    except Exception as e:
        print(f" X {ticker}: {e}")

if all_news:
    df_bronze_news = spark.createDataFrame(all_news)
    df_bronze_news.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.bronze_news")
    print(f"\nBronze news: {len(all_news)} articles written")


In [0]:

# --- Silver: Clean and type price data ---
df_silver_prices = (
    spark.table(f"{CATALOG}.{SCHEMA}.bronze_prices")
    .select(
        F.col("ticker"),
        F.to_date(F.col("date")).alias("snapshot_date"),
        F.col("open").cast("decimal(12,4)").alias("open_price"),
        F.col("close").cast("decimal(12,4)").alias("close_price"),
        F.col("high").cast("decimal(12,4)").alias("high_price"),
        F.col("low").cast("decimal(12,4)").alias("low_price"),
        F.col("volume").cast("bigint")
    )
    .dropDuplicates(["ticker", "snapshot_date"])
    .filter(F.col("snapshot_date").isNotNull())
)

df_silver_prices.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.silver_prices")
print(f"Silver prices: {df_silver_prices.count()} rows")
df_silver_prices.show(5)

In [0]:
# --- Silver: Standardize company profiles ---
df_silver_companies = (
    spark.table(f"{CATALOG}.{SCHEMA}.bronze_companies")
    .select(
        F.col("ticker"),
        F.col("name").alias("company_name"),
        F.col("sector"),
        F.col("industry"),
        F.col("marketCap").cast("bigint").alias("market_cap"),
        F.col("description").alias("company_description"),
        F.current_timestamp().alias("updated_at")
    )
    .dropDuplicates(["ticker"])
)

df_silver_companies.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.silver_companies")
print(f"Silver companies: {df_silver_companies.count()} rows")
df_silver_companies.show(5, truncate=40)




In [0]:

# --- Silver: Clean news articles ---
df_silver_news = (
    spark.table(f"{CATALOG}.{SCHEMA}.bronze_news")
    .select(
        F.col("ticker"),
        F.col("title"),
        F.col("source"),
        F.to_timestamp(F.col("publishedAt")).alias("published_at"),
        F.col("url"),
        F.col("summary"),
        F.col("content").alias("full_text"),
        F.current_timestamp().alias("ingested_at")
    )
    .filter(F.col("title").isNotNull())
)

df_silver_news.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.silver_news")
print(f"Silver news: {df_silver_news.count()} rows")
df_silver_news.show(5, truncate=40)



In [0]:

'''
# --- Optional: Sync Silver tables to Lakebase for low-latency app reads ---
# This uses OAuth authentication with Lakebase Postgres.
# Note: Your app.py already uses OAuth tokens, so this manual sync is optional.
# Consider using Lakebase Synced Tables for automatic sync instead.

from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
LAKEBASE_ENDPOINT = dbutils.secrets.get(scope="stock-research", key="lakebase-endpoint")

# Generate OAuth token for Postgres connection
token_response = w.postgres.generate_database_credential(endpoint=LAKEBASE_ENDPOINT)

# Extract host from endpoint (format: projects/name/branches/branch/endpoints/primary)
LAKEBASE_PROJECT = dbutils.secrets.get(scope="stock-research", key="lakebase-project")
LAKEBASE_BRANCH = dbutils.secrets.get(scope="stock-research", key="lakebase-branch")
LAKEBASE_HOST = f"{LAKEBASE_PROJECT}-{LAKEBASE_BRANCH}.cloud.databricks.com"  # Adjust based on your actual host format

jdbc_url = f"jdbc:postgresql://{LAKEBASE_HOST}:5432/databricks_postgres?sslmode=require"
jdbc_props = {
    "user": "oauth",
    "password": token_response.password,
    "driver": "org.postgresql.Driver"
}

# Push price snapshots
df_silver_prices.write.jdbc(
    url=jdbc_url,
    table="price_snapshots",
    mode="overwrite",
    properties=jdbc_props
)
print("✓ Synced price_snapshots to Lakebase")

# Push companies
df_silver_companies.write.jdbc(
    url=jdbc_url,
    table="companies",
    mode="overwrite",
    properties=jdbc_props
)
print("✓ Synced companies to Lakebase")
print("\nPipeline complete.")
'''